# Requisitos e Objetivos do Projeto

**Disciplina:** Engenharia de Software para IA e Frameworks Profundos
**Projeto:** Análise de sentimentos de avaliações de produtos
**Entrega 4:** Testes automatizados (unittest) e documento de requisitos

Este notebook descreve o objetivo, os requisitos e a estrutura do projeto.
Ele serve como referência do que o sistema faz e de como cada requisito é
verificado pela suíte de testes.

## 1. Objetivo geral

Construir uma aplicação de análise de sentimentos que classifica avaliações
de produtos como positivas ou negativas, aplicando práticas de engenharia de
software: modularização, tipagem, testes automatizados e versionamento com Git.
O modelo é uma regressão logística treinada com PyTorch sobre uma matriz de
contagem de palavras construída com NumPy.

## 2. Objetivos específicos

1. Carregar o conjunto de avaliações a partir de um arquivo CSV e validar suas colunas.
2. Pré-processar o texto (normalização) e derivar um rótulo binário a partir da nota.
3. Representar cada avaliação como um vetor de contagem de palavras.
4. Separar os dados em treino e teste de forma reprodutível.
5. Treinar o modelo em PyTorch e imprimir o erro de treino e de teste.
6. Avaliar o modelo com acurácia, precisão, revocação e F1.
7. Persistir o modelo treinado e as métricas em disco.
8. Garantir a qualidade do código com uma suíte de testes automatizados em `unittest`.

## 3. Descrição do problema e dos dados

A entrada é um conjunto de avaliações de produtos da Amazon (Kaggle, autor `yasserh`),
com uma coluna de texto (`reviews.text`) e uma coluna de nota de 1 a 5 (`reviews.rating`).

O rótulo de sentimento é derivado da nota:

| Nota | Sentimento | Rótulo |
|------|------------|--------|
| 4 ou 5 | positivo | 1 |
| 1 ou 2 | negativo | 0 |
| 3 | neutro | descartado |

## 4. Requisitos funcionais

Descrevem o que o sistema faz. Cada requisito aponta para a função que o implementa.

| ID | Requisito | Implementação |
|----|-----------|---------------|
| RF01 | Carregar o CSV, validar colunas obrigatórias e rejeitar arquivo inexistente ou vazio | `src/data/loader.py`: `load_data`, `validate_columns` |
| RF02 | Normalizar o texto (minúsculas, remoção de pontuação, colapso de espaços) | `src/preprocessing/transform.py`: `clean_text` |
| RF03 | Converter a nota em rótulo binário e descartar avaliações neutras | `src/preprocessing/transform.py`: `normalize_label`, `preprocess_dataset` |
| RF04 | Construir o vocabulário e a matriz de contagem de palavras | `src/preprocessing/transform.py`: `build_vocabulary`, `texts_to_matrix` |
| RF05 | Separar treino e teste de forma reprodutível | `src/training/train.py`: `split_dataset` |
| RF06 | Treinar o modelo e imprimir o erro de treino e de teste | `src/models/model.py`: `train_model` |
| RF07 | Prever o rótulo de novas avaliações | `src/models/model.py`: `predict` |
| RF08 | Avaliar o modelo com acurácia, precisão, revocação e F1 | `src/evaluation/metrics.py`: `evaluate_model` |
| RF09 | Persistir o modelo treinado e as métricas | `src/models/model.py`: `save_model`; `src/training/train.py`: `save_metrics` |

## 5. Requisitos não funcionais

Descrevem restrições de qualidade do sistema.

| ID | Requisito | Como é atendido |
|----|-----------|-----------------|
| RNF01 | Reprodutibilidade | Semente fixa `RANDOM_SEED = 42` na divisão dos dados e no treino |
| RNF02 | Modularidade | Código separado por responsabilidade em `src/` (data, preprocessing, models, training, evaluation, utils) |
| RNF03 | Tipagem | Anotações de tipo em todas as assinaturas públicas |
| RNF04 | Portabilidade de dispositivo | Código agnóstico de dispositivo (CPU ou GPU CUDA) |
| RNF05 | Testabilidade | Suíte `unittest` cobrindo módulos e pipeline, executável sem dependências extras |
| RNF06 | Legibilidade | Código em inglês, docstrings e estilo PEP 8 |
| RNF07 | Versionamento | Git com entregas incrementais |

## 6. Pipeline do projeto

As etapas do sistema, na ordem em que são executadas:

```
carregar -> pré-processar -> vetorizar -> dividir -> treinar -> avaliar -> salvar
```

- **Carregar** (`load_data`): lê o CSV e valida as colunas.
- **Pré-processar** (`preprocess_dataset`): limpa o texto e deriva o rótulo.
- **Vetorizar** (`build_vocabulary`, `texts_to_matrix`): produz a matriz de contagem.
- **Dividir** (`split_dataset`): separa 80% treino e 20% teste.
- **Treinar** (`train_model`): ajusta os pesos e imprime o erro de treino e de teste.
- **Avaliar** (`evaluate_model`): calcula as métricas no conjunto de teste.
- **Salvar** (`save_model`, `save_metrics`): grava o modelo e as métricas.

## 7. Estrutura de módulos

```
src/
├── data/loader.py            # RF01
├── preprocessing/transform.py # RF02, RF03, RF04
├── models/model.py           # RF06, RF07, RF09
├── training/train.py         # RF05, RF09 e orquestração do pipeline
├── evaluation/metrics.py     # RF08
└── utils/config.py           # constantes e parâmetros
tests/                        # suíte unittest (Entrega 4)
```

## 8. Escopo

**Dentro do escopo:**

- Classificação binária de sentimento (positivo ou negativo) a partir do texto da avaliação.
- Pipeline completo: carga, pré-processamento, vetorização, treino, avaliação e persistência.
- Representação por contagem de palavras (bag of words) e modelo de regressão logística.

**Fora do escopo:**

- Avaliações neutras (nota 3), que são descartadas.
- Interface gráfica ou API de serviço.
- Modelos de linguagem pré-treinados ou embeddings semânticos.
- Idiomas além do presente no conjunto de dados.

## 9. Restrições e limitações

- O rótulo é derivado da nota, e não de uma anotação humana do sentimento.
- O vocabulário é construído apenas com as palavras do conjunto de treino;
  palavras novas no teste são ignoradas.
- A representação por contagem de palavras ignora ordem e contexto.
- O desempenho depende do equilíbrio entre avaliações positivas e negativas na base.

## 10. Critérios de aceitação

- O carregamento rejeita arquivo inexistente e conjunto vazio (RF01).
- O pré-processamento produz texto normalizado e rótulo binário; a nota 3 é descartada (RF02, RF03).
- A divisão treino/teste conserva o total de linhas e é reprodutível com semente fixa (RF05, RNF01).
- O treino imprime o erro de treino e de teste ao longo das épocas (RF06).
- A previsão retorna apenas rótulos 0 ou 1, com o mesmo tamanho da entrada (RF07).
- O modelo salvo pode ser recarregado e reproduz as mesmas previsões (RF09).
- Toda a suíte de testes passa com `python -m unittest discover -s tests`.

## 11. Rastreabilidade: requisitos e testes

Matriz de rastreabilidade ligando cada requisito funcional a pelo menos um teste em `tests/`.

| Requisito | Arquivo de teste |
|-----------|------------------|
| RF01 | `tests/test_data_loader.py` |
| RF02, RF03 | `tests/test_preprocessing.py` |
| RF04, RF05 | `tests/test_properties.py`, `tests/test_split.py` |
| RF06, RF07 | `tests/test_predict.py`, `tests/test_model.py` |
| RF08 | `tests/test_evaluation.py` |
| RF09 (salvar e carregar) | `tests/test_model.py` |
| RF01-RF09 (fim a fim) | `tests/test_integration.py` |

## 12. Verificação

As células abaixo confirmam que os módulos carregam e que a suíte de testes passa.
A suíte usa apenas a biblioteca padrão (`unittest`), sem dependências extras.

In [ ]:
# As funções públicas de cada requisito estão disponíveis para importação.
from src.data.loader import load_data, validate_columns
from src.preprocessing.transform import (
    clean_text,
    normalize_label,
    build_vocabulary,
    texts_to_matrix,
)
from src.training.train import split_dataset, run_training
from src.models.model import train_model, predict, save_model
from src.evaluation.metrics import evaluate_model

print("Módulos importados com sucesso.")

In [ ]:
# Exemplo mínimo: normalização de texto e derivação de rótulo (RF02, RF03).
print(clean_text("Excelente produto, recomendo!!!"))
print(normalize_label(5), normalize_label(2), normalize_label(3))

In [ ]:
# Execução da suíte de testes (Entrega 4).
import unittest

suite = unittest.TestLoader().discover("tests")
unittest.TextTestRunner().run(suite)